# VinDr-CXR: CLEAR vs CheXzero within images with and without recorded demographics

CPU only. Inputs: the VinBigData scoring notebook output. Image-level bootstrap, 1,000 resamples, seed 2026.

In [ ]:
import os, json, time, numpy as np, pandas as pd
T0 = time.time(); SEED = 2026
def find(name, must=None):
    hits = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if name in files and (must is None or must in root): hits.append(os.path.join(root, name))
    return sorted(hits)
vin_s = find("scores_chexzero.npy", "vin_out")
assert vin_s, "VinBigData output (vin_out) not found: add that notebook as input"
vin_dir = os.path.dirname(vin_s[0]); print("Vin:", vin_dir)
FINDINGS = ["Atelectasis","Cardiomegaly","Effusion","Infiltration","Mass","Nodule","Pneumonia",
            "Pneumothorax","Consolidation","Edema","Emphysema","Fibrosis","Pleural_Thickening","Hernia"]
class Cell:
    def __init__(self, s, y, pidx):
        o = np.argsort(s, kind="mergesort"); s = s[o]; self.y, self.p = y[o].astype(np.float64), pidx[o]
        self.starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    def auc(self, w=None):
        wi = 1.0 if w is None else w[self.p]
        bp = np.add.reduceat(wi * self.y, self.starts); bn = np.add.reduceat(wi * (1 - self.y), self.starts)
        P, N = bp.sum(), bn.sum()
        return np.nan if P == 0 or N == 0 else float((bp * (np.cumsum(bn) - bn + 0.5 * bn)).sum() / (P * N))
def ci(a, lvl=0.95):
    a = a[~np.isnan(a)]
    return (float("nan"), float("nan")) if a.size == 0 else (float(np.quantile(a, (1-lvl)/2)), float(np.quantile(a, 1-(1-lvl)/2)))
def table(s, Y, mask, pidx, WB):
    idx = np.flatnonzero(mask); pt, bt = [], []
    for j in range(Y.shape[1]):
        c = Cell(s[idx, j], Y[idx, j], pidx[idx]); pt.append(c.auc()); bt.append([c.auc(w) for w in WB])
    return np.array(pt), np.array(bt).T
def mac(tab, E): pt, bt = tab; return float(np.nanmean(pt[E])), np.nanmean(bt[:, E], axis=1)
def fmt(p, c): return f"{p:+.3f} ({c[0]:+.3f} to {c[1]:+.3f})"
OUT = {}

In [ ]:

vc = pd.read_csv(os.path.join(vin_dir, "cohort.csv"))
VF = [c for c in vc.columns if c not in ("image_id", "sex", "age", "age_band4") and not c.startswith("any_")]
VS = {k: np.load(os.path.join(vin_dir, f"scores_{k}.npy")).astype(np.float64) for k in ["chexzero", "clear_concept", "clear_direct"]}
Yv = vc[VF].values; n = len(vc); iidx = np.arange(n)
rng = np.random.default_rng(SEED); WV = [np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float64) for _ in range(1000)]
sk = vc.sex.isin(["M", "F"]).values; ak = vc.age.notna().values
subsets = {"all": np.ones(n, bool), "sex_known": sk, "sex_missing": ~sk, "age_known": ak, "age_missing": ~ak,
           "both_known": sk & ak, "both_missing": ~sk & ~ak}
print("===== VINDR SUBSET SUMMARY =====")
print("n:", {g: int(m.sum()) for g, m in subsets.items()})
print("normal-image share (no eligible finding):", {g: round(float((Yv[m].sum(1) == 0).mean()), 3) for g, m in subsets.items()})
res = {}
for g, m in subsets.items():
    E = [j for j in range(len(VF)) if Yv[m, j].sum() >= 30]
    if not E: print(f"-- {g}: no finding with 30 positives"); continue
    T = {k: table(VS[k], Yv, m, iidx, WV) for k in VS}
    pc, bc = mac(T["chexzero"], E); line = f"-- {g} (n={int(m.sum())}, {len(E)} findings): CheXzero {pc:.3f}"
    for k in ["clear_concept", "clear_direct"]:
        pk, bk = mac(T[k], E); line += f" | {k} {pk:.3f}, minus CheXzero {fmt(pk - pc, ci(bk - bc))}, wins/losses {int((T[k][0][E] > T['chexzero'][0][E]).sum())}/{int((T[k][0][E] < T['chexzero'][0][E]).sum())}"
    print(line)
    res[g] = {VF[j]: [round(float(T[k][0][j]), 3) for k in ["chexzero", "clear_concept", "clear_direct"]] for j in E}
for g in ["sex_known", "sex_missing"]:
    if g in res: print(f"   per finding {g} (CheXzero / concept / direct):", res[g])
print("minutes:", round((time.time() - T0) / 60, 1))
print("===== END =====")
